In [ ]:
#%% md
# Ноутбук 3: LLM-валидация кластеров дублей + Cluster Stability Assessment Index (CSAI)
#
# Гипотеза G: качество кластеров дублей, найденных обычным cosine-threshold методом,
#     можно проверить и уточнить с помощью независимого "судьи" — модели, которая явно
#     оценивает пару текстов на предмет того, что это ОДНА И ТА ЖЕ новость (а не просто похожий
#     набор слов). В идеале это большая LLM (через API); здесь используется лёгкий открытый
#     cross-encoder NLI как локальный прокси для LLM-судьи — тот же принцип, просто без
#     затрат на внешний API. При наличии ключа Anthropic/OpenAI функцию llm_judge_pair()
#     легко заменить на реальный вызов LLM (см. комментарий в коде).
#
# Метрика CSAI (наша интерпретация): устойчивое разбиение на дубли должно давать хороший
#     silhouette НЕЗАВИСИМО от того, в каком признаковом пространстве его оценивать (word
#     TF-IDF или char n-gram). Если кластер хорош только в одном пространстве — это
#     подозрительно (возможно, ложное срабатывание конкретного метода). Важно: это наша
#     собственная интерпретация идеи из статьи, а не официальная опубликованная формула.
#
# НЕ включаем сюда: выбор конкретной модели эмбеддингов (ноутбук 2), экономический анализ
# (ноутбук 1), потоковую обработку (ноутбук 4) — здесь фокус только на проверке качества
# уже готового кластеринга.
#%%


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import silhouette_score
from tqdm.auto import tqdm

DATA_PATH = '/kaggle/input/datasets/artemwalker/telegram-news/telegram_news_raw.csv'
OUT_DIR = Path('llm_cluster_validation_results')
OUT_DIR.mkdir(exist_ok=True)
WINDOW_DAYS = 14

df = pd.read_csv(DATA_PATH, parse_dates=['post_dttm']).reset_index(drop=True)

def normalize_text(text):
    text = str(text or '').lower()
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'@[\w_]+', ' ', text)
    text = re.sub(r'[^\w\s%.-]', ' ', text, flags=re.UNICODE)
    return re.sub(r'\s+', ' ', text).strip()

df['text_normalized'] = df['data'].fillna('').map(normalize_text)
df = df[df.text_normalized.str.len().gt(0)].copy().reset_index(drop=True)

class UnionFind:
    def __init__(self, n): self.parent = list(range(n))
    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x
    def union(self, a, b):
        a, b = self.find(a), self.find(b)
        if a != b: self.parent[b] = a

def clusters_from_edges(n, edges):
    uf = UnionFind(n)
    for a, b in edges: uf.union(a, b)
    return pd.factorize([uf.find(i) for i in range(n)])[0]

def top_k_edges(vectors, dates, threshold, n_neighbors=6, batch_size=128):
    ts = pd.to_datetime(dates, utc=True).astype('int64').to_numpy()
    max_delta = pd.Timedelta(days=WINDOW_DAYS).value
    edges = []
    for start in range(0, len(ts), batch_size):
        stop = min(start + batch_size, len(ts))
        sim = vectors[start:stop] @ vectors.T
        if hasattr(sim, 'toarray'): sim = sim.toarray()
        idx = np.argpartition(sim, -n_neighbors, axis=1)[:, -n_neighbors:]
        for local_i, candidates in enumerate(idx):
            i = start + local_i
            for j in candidates:
                if i < j and sim[local_i, j] >= threshold and abs(ts[i] - ts[j]) <= max_delta:
                    edges.append((i, int(j)))
    return edges

THRESHOLD = 0.85  # намеренно немного мягче базового 0.90, чтобы было что валидировать
tfidf = TfidfVectorizer(analyzer='word', ngram_range=(2, 5), min_df=4, max_features=500_000)
tfidf_vectors = tfidf.fit_transform(df.text_normalized)
candidate_edges = top_k_edges(tfidf_vectors, df.post_dttm, THRESHOLD)
candidate_clusters = clusters_from_edges(len(df), candidate_edges)
print(f'Кандидатных пар-дублей (TF-IDF >= {THRESHOLD}): {len(candidate_edges):,}')

In [ ]:
#%% md
### Гипотеза G: LLM/cross-encoder судья для проверки пар-кандидатов
# Как применяем: на каждую кандидатную пару из candidate_edges просим "судью" оценить,
#     это одна и та же новость или нет (бинарно), используя открытую NLI cross-encoder модель
#     как локальный прокси LLM-вердикта. Оцениваем случайную выборку пар (не все — дорого),
#     чтобы получить оценку доли ложных срабатываний по всему множеству кандидатов.
# Как оцениваем результат: доля пар, подтверждённых судьёй ("оценка precision"), с
#     доверительным интервалом на размере выборки.
#%%
try:
    from sentence_transformers import CrossEncoder
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'sentence-transformers'])
    from sentence_transformers import CrossEncoder

try:
    # Прокси LLM-судьи: мультиязычная NLI cross-encoder модель.
    judge = CrossEncoder('cross-encoder/nli-deberta-v3-base')
    judge_available = True
except Exception as exc:
    print(f'Не удалось загрузить cross-encoder ({exc}) — гипотеза G будет пропущена.')
    judge_available = False

def llm_judge_pair(text_a, text_b):
    """
    Прокси LLM-судьи. Чтобы использовать настоящую LLM вместо cross-encoder, замените тело
    функции на вызов API (Anthropic/OpenAI) с промптом вида:
    "Это одна и та же новость? Ответь только 'да' или 'нет'.\n\nНовость 1: ...\n\nНовость 2: ..."
    и распарсите ответ в bool.
    """
    scores = judge.predict([(text_a, text_b)])
    # nli-deberta возвращает [contradiction, entailment, neutral] логиты — берём entailment
    return int(np.argmax(scores[0])) == 1

if judge_available and len(candidate_edges) > 0:
    SAMPLE_SIZE = min(200, len(candidate_edges))
    rng = np.random.default_rng(42)
    sample_idx = rng.choice(len(candidate_edges), size=SAMPLE_SIZE, replace=False)

    verdicts = []
    for k in tqdm(sample_idx, desc='LLM-проверка выборки пар'):
        i, j = candidate_edges[k]
        verdicts.append(llm_judge_pair(df.text_normalized.iloc[i], df.text_normalized.iloc[j]))

    precision_hat = float(np.mean(verdicts))
    se = np.sqrt(precision_hat * (1 - precision_hat) / SAMPLE_SIZE)
    print(f'Оценка precision по выборке из {SAMPLE_SIZE} пар: {precision_hat:.1%} +/- {1.96*se:.1%} (95% ДИ)')

    pd.DataFrame({'edge_idx': sample_idx, 'confirmed_by_judge': verdicts}).to_csv(
        OUT_DIR / 'llm_judge_sample.csv', index=False, encoding='utf-8-sig')
else:
    print('Гипотеза G пропущена: нет кандидатных пар или судья недоступен.')

In [ ]:
#%% md
### Метрика CSAI (наша интерпретация): устойчивость кластеров в разных признаковых пространствах
# Как применяем: берём то же самое разбиение candidate_clusters, но считаем silhouette
#     в ДВУХ независимых признаковых пространствах — word TF-IDF (в котором кластеры и
#     искались) и char n-gram TF-IDF (как суррогат независимого представления, чтобы не
#     тянуть тяжёлый трансформер только ради этой проверки). CSAI = среднее двух silhouette,
#     разница между ними = индикатор нестабильности (кластер хорош только в одном пространстве).
# Как оцениваем результат: чем выше CSAI и чем МЕНЬШЕ разница между двумя silhouette —
#     тем устойчивее найденное разбиение на дубли.
#%%
dup_sizes = pd.Series(candidate_clusters).value_counts()
dup_ids = dup_sizes[dup_sizes > 1].index
dup_mask = np.isin(candidate_clusters, dup_ids)

char_tfidf = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=4, max_features=200_000)
char_vectors = char_tfidf.fit_transform(df.text_normalized)

if dup_mask.sum() > 1 and len(dup_ids) > 1:
    sil_word = silhouette_score(tfidf_vectors[dup_mask], candidate_clusters[dup_mask], metric='cosine')
    sil_char = silhouette_score(char_vectors[dup_mask], candidate_clusters[dup_mask], metric='cosine')
    csai = (sil_word + sil_char) / 2
    instability = abs(sil_word - sil_char)
    print(f'Silhouette (word TF-IDF) = {sil_word:.3f}')
    print(f'Silhouette (char n-gram) = {sil_char:.3f}')
    print(f'CSAI (среднее) = {csai:.3f}; нестабильность (|разница|) = {instability:.3f}')

    pd.DataFrame([{
        'silhouette_word': sil_word, 'silhouette_char': sil_char,
        'csai': csai, 'instability': instability
    }]).to_csv(OUT_DIR / 'csai_report.csv', index=False, encoding='utf-8-sig')
else:
    print('Слишком мало кластеров-дублей для расчёта CSAI.')

In [ ]:
# Итог ноутбука 3:
#  - llm_judge_sample.csv — вердикты LLM/cross-encoder-судьи по выборке пар (гипотеза G)
#  - csai_report.csv — метрика устойчивости кластеров в двух признаковых пространствах
print('Готово. Файлы результатов сохранены в', OUT_DIR.resolve())